# Coupling 4변형 비교 — 전역/지역 × Pearson/Spearman

MSN–FC coupling을 정의하는 네 가지 방식을 **같은 fold, 같은 모델**로 한 번에 돌린다.
노트북을 나누면 fold가 달라져 차이가 변형 때문인지 우연인지 구분되지 않는다.

| 변형 | 정의 | 피처 수 |
|---|---|---|
| `Coup_glob_P` | 상삼각 2278쌍 전체의 **Pearson** | 1 |
| `Coup_glob_S` | 상삼각 2278쌍 전체의 **Spearman** | 1 |
| `Coup_reg_P` | ROI마다 67쌍 **Pearson** → PCA 3 | 68 → 3 |
| `Coup_reg_S` | ROI마다 67쌍 **Spearman** → PCA 3 | 68 → 3 |

`Coup_reg_P`는 `automl_multimodal_MSN_FC68_coupling.ipynb`와 같은 정의다(재현 확인용).

## 미리 확인된 사실 (vas를 보지 않고 계산)

네 변형은 서로 매우 닮았다. 피험자 간 상관이

- 전역 Pearson vs Spearman: **r = 0.963**
- 지역 Pearson vs Spearman: **r = 0.953** (ROI 평균)
- 전역 vs 지역평균: **r = 0.97 이상**

따라서 Pearson/Spearman 차이는 **다른 분석이 아니라 민감도 확인**으로 읽어야 한다.
셋 이상에서 같은 방향이면 신호, 하나만 튀면 노이즈다.

## Spearman을 같이 보는 이유

MSN edge는 measure 8개로 만든 상관이라 |r| > 0.9가 7.9%로 흔하다.
순위 상관은 극단값에 덜 휘둘리고, **단조 변환에 불변**이라 FC를 r로 쓰든 Fisher z로 쓰든 값이 같다.
PNAS 2024(청소년 MSN–FC coupling)도 전역·지역 모두 Spearman을 썼다.

## 해석 규칙

네 개를 다 돌리는 만큼, **좋은 것만 골라 보고하면 그 자체가 leakage**다.
네 변형 결과를 전부 보고하고, 판단은 "몇 개에서 일관되게 이득이 나왔나"로 한다.
PCA 성분 수(3)는 기존 노트북과 맞춘 사전 고정값이며 결과를 보고 바꾸지 않는다.

## treatment_group 스위치

`USE_TREATMENT_GROUP = False`가 기본이다. 모델이 치료군을 모른 채 학습하고,
Active/Sham은 사후 비교에만 쓴다. 이렇게 해야 군 라벨 순열검정이 가능하다(5번 셀).
기존 노트북과 같은 조건으로 맞추려면 `True`로 바꾼 뒤 전체 재실행.

In [ ]:
import os
# joblib 병렬 처리시 비-ASCII 경로에서 UnicodeEncodeError 나는 것 회피
os.environ['JOBLIB_TEMP_FOLDER'] = r'C:\SUDMEX\jltmp'
os.makedirs(os.environ['JOBLIB_TEMP_FOLDER'], exist_ok=True)

import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)

In [ ]:
# ==========================================================
# 1. DATA PREPARATION — coupling 4변형 계산
# ==========================================================
DATA_DIR = '..'
USE_TREATMENT_GROUP = False   # False면 모델은 치료군을 모른다 (사후 비교 전용)

def load(name):
    d = pd.read_csv(os.path.join(DATA_DIR, name), skipinitialspace=True)
    d.columns = d.columns.str.strip()
    return d

msn_s = load('msn_strength_68.csv')   # 앞 7열: rid, treatment_group, vas_t0, vas_t1, age, sex, eTIV
fc_s  = load('fc_strength_68.csv')    # 앞 7열: ... mean_fd
msn_e = load('msn_full_68x68.csv')
fc_e  = load('fc_full_68x68.csv')

# --- ROI 매칭 확인: 접미사만 떼면 이름과 순서가 같아야 한다 ---
ROIS = [c[:-len('_msn')] for c in msn_s.columns[7:]]
assert len(ROIS) == 68, len(ROIS)
assert [c[:-len('_fc')] for c in fc_s.columns[7:]] == ROIS, 'strength ROI 순서 불일치'
msn_edge_cols, fc_edge_cols = list(msn_e.columns[7:]), list(fc_e.columns[7:])
assert [c[:-len('_msn')] for c in msn_edge_cols] == fc_edge_cols, 'edge 순서 불일치'

# --- rid inner join: 공통 44명 ---
df = (msn_s.merge(fc_s[['rid', 'mean_fd']], on='rid', how='inner')
      .sort_values('rid').reset_index(drop=True))
assert len(df) == 44, len(df)

E_msn = msn_e.set_index('rid').loc[df['rid'], msn_edge_cols].values   # (44, 2278)
E_fc  = fc_e.set_index('rid').loc[df['rid'], fc_edge_cols].values

IU  = np.triu_indices(68, k=1)
OFF = ~np.eye(68, dtype=bool)

def to_matrix(edge_row):
    M = np.zeros((68, 68))
    M[IU] = edge_row
    return M + M.T          # 대각은 OFF로 빼고 쓰므로 0으로 둔다

# ---- 전역 coupling: 상삼각 2278쌍을 통째로 → 사람당 숫자 1개 ----
X_gP = np.array([[pearsonr(E_msn[k], E_fc[k])[0]] for k in range(len(df))])
X_gS = np.array([[spearmanr(E_msn[k], E_fc[k]).statistic] for k in range(len(df))])

# ---- 지역 coupling: ROI마다 자기 제외 67쌍 → 사람당 68개 ----
X_rP = np.zeros((len(df), 68))
X_rS = np.zeros((len(df), 68))
for k in range(len(df)):
    S, F = to_matrix(E_msn[k]), to_matrix(E_fc[k])
    for i in range(68):
        a, b = S[i, OFF[i]], F[i, OFF[i]]
        X_rP[k, i] = pearsonr(a, b)[0]
        X_rS[k, i] = spearmanr(a, b).statistic

for nm, X in [('gP', X_gP), ('gS', X_gS), ('rP', X_rP), ('rS', X_rS)]:
    assert np.all(np.isfinite(X)), nm

# ---- nuisance / clinical / target ----
X_nuis = df[['age', 'sex', 'eTIV', 'mean_fd']].values   # coupling은 두 모달리티의 교란을 다 받는다
clinical_cols = ['vas_t0'] + (['treatment_group'] if USE_TREATMENT_GROUP else [])
X_clinical = df[clinical_cols].values
y = df['vas_t1'].values
group = np.where(df['treatment_group'].values == 2, 'Active', 'Sham')

print(f'n = {len(df)}  |  Active {(group == "Active").sum()} / Sham {(group == "Sham").sum()}')
print(f'임상 피처: {clinical_cols}   (USE_TREATMENT_GROUP={USE_TREATMENT_GROUP})')
print()
for nm, X in [('전역 Pearson ', X_gP), ('전역 Spearman', X_gS),
              ('지역 Pearson ', X_rP), ('지역 Spearman', X_rS)]:
    print(f'  {nm}  평균 {X.mean():+.3f}  SD {X.std():.3f}  '
          f'범위 {X.min():+.3f} ~ {X.max():+.3f}')

In [ ]:
# ==========================================================
# 2. MODELS & SETUP  (기존 노트북과 동일)
# ==========================================================
models = {
    'LinearRegression': LinearRegression(),
    'ElasticNet': ElasticNet(max_iter=10000, random_state=42),
    'SVR_Linear': SVR(kernel='linear'),
    'SVR_RBF': SVR(kernel='rbf'),
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42, objective='reg:squarederror')
}

param_grids = {
    'LinearRegression': {},
    'ElasticNet': {'alpha': [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0],
                   'l1_ratio': [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9]},
    'SVR_Linear': {'C': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
                   'epsilon': [0.01, 0.1, 0.5, 1.0, 2.0]},
    'SVR_RBF': {'C': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
                'gamma': ['scale', 0.0001, 0.001, 0.01, 0.1],
                'epsilon': [0.01, 0.1, 0.5, 1.0, 2.0]},
    'RandomForest': {'n_estimators': [100, 300], 'max_depth': [None, 2, 3, 5],
                     'min_samples_leaf': [1, 3, 5]},
    'XGBoost': {'n_estimators': [100, 300], 'max_depth': [2, 3],
                'learning_rate': [0.03, 0.1], 'reg_lambda': [1, 10]},
}

N_PCA = 3   # 사전 고정 (기존 멀티모달 노트북과 동일)

# 이름 -> (원자료, PCA를 쓸지)
VARIANTS = {
    'Coup_glob_P': (X_gP, False),
    'Coup_glob_S': (X_gS, False),
    'Coup_reg_P':  (X_rP, True),
    'Coup_reg_S':  (X_rS, True),
}
DATA_TYPES = ['Clinical_ONLY'] + list(VARIANTS)

In [ ]:
# ==========================================================
# 3. CROSS-VALIDATION (LOOCV)
#    fold 안에서: nuisance 잔차 -> (지역이면) 표준화 + PCA3 -> 임상과 결합 -> 표준화
#    ※ 5 datasets x 6 models x 44 folds — 수 분 ~ 수십 분
# ==========================================================
def resid(X, N, tr, te):
    # nuisance 회귀 잔차. 회귀는 train에서만 fit한다.
    reg = LinearRegression().fit(N[tr], X[tr])
    return X[tr] - reg.predict(N[tr]), X[te] - reg.predict(N[te])

def prep(X, use_pca, tr, te):
    # 전역(1개)은 잔차만, 지역(68개)은 잔차 -> 표준화 -> PCA3
    Xtr, Xte = resid(X, X_nuis, tr, te)
    if not use_pca:
        return Xtr, Xte, np.nan
    sc = StandardScaler().fit(Xtr)
    pca = PCA(n_components=N_PCA, random_state=42).fit(sc.transform(Xtr))
    return (pca.transform(sc.transform(Xtr)), pca.transform(sc.transform(Xte)),
            pca.explained_variance_ratio_.sum())

loo = LeaveOneOut()
predictions     = {(dt, name): [] for dt in DATA_TYPES for name in models}
best_params_log = {(dt, name): [] for dt in DATA_TYPES for name in models}
evr_log = {k: [] for k, (_, p) in VARIANTS.items() if p}
y_true = []

print(f"Starting LOOCV: {len(models)} models x {len(DATA_TYPES)} datasets x {len(y)} folds ...\n")

for fold_idx, (tr, te) in enumerate(loo.split(y)):
    print(f"fold {fold_idx + 1}/{len(y)}", end='\r')
    y_tr = y[tr]
    y_true.append(y[te][0])
    Xc_tr, Xc_te = X_clinical[tr], X_clinical[te]

    feat_sets = {'Clinical_ONLY': (Xc_tr, Xc_te)}
    for nm, (X, use_pca) in VARIANTS.items():
        Xb_tr, Xb_te, evr = prep(X, use_pca, tr, te)
        if use_pca:
            evr_log[nm].append(evr)
        feat_sets[nm] = (np.hstack([Xb_tr, Xc_tr]), np.hstack([Xb_te, Xc_te]))

    for dt in DATA_TYPES:
        Xtr_raw, Xte_raw = feat_sets[dt]
        sc = StandardScaler().fit(Xtr_raw)
        Xtr, Xte = sc.transform(Xtr_raw), sc.transform(Xte_raw)
        for name, model in models.items():
            gcv = GridSearchCV(model, param_grids[name], cv=3,
                               scoring='neg_mean_squared_error', n_jobs=-1)
            gcv.fit(Xtr, y_tr)
            predictions[(dt, name)].append(float(gcv.best_estimator_.predict(Xte)[0]))
            best_params_log[(dt, name)].append(str(gcv.best_params_))

y_true = np.array(y_true)
print("\n\nLOOCV done.")
print('PCA3 누적 설명분산 평균: ' +
      '  |  '.join(f'{k} {np.mean(v):.1%}' for k, v in evr_log.items()))

In [ ]:
# ==========================================================
# 4. EVALUATION — 전체 성능
# ==========================================================
def metrics(yt, yp):
    r, p = pearsonr(yt, yp)
    return dict(RMSE=np.sqrt(mean_squared_error(yt, yp)), R2=r2_score(yt, yp), r=r, p=p)

rows = []
for (dt, name), preds in predictions.items():
    yp = np.array(preds)
    for scope, m in [('Overall', np.ones(len(y_true), bool)),
                     ('Active', group == 'Active'), ('Sham', group == 'Sham')]:
        rows.append({'Data Type': dt, 'Model': name, 'Scope': scope,
                     'n': int(m.sum()), **metrics(y_true[m], yp[m])})
res = pd.DataFrame(rows)

baseline_rmse = np.sqrt(np.mean([(y_true[i] - np.delete(y_true, i).mean()) ** 2
                                 for i in range(len(y_true))]))
print(f"평균만 예측하는 모델 LOOCV RMSE = {baseline_rmse:.3f}  (n={len(y_true)})")
print("  -> 이 값보다 나쁜 모델은 예측력 없음\n")

ov = (res[res.Scope == 'Overall'].pivot(index='Model', columns='Data Type', values='RMSE')
      [DATA_TYPES].loc[list(models)])
for nm in VARIANTS:
    ov[f'd_{nm}'] = ov['Clinical_ONLY'] - ov[nm]      # + 면 coupling이 이득

print("========== Overall RMSE (낮을수록 좋음) / d = 양수면 이득 ==========")
display(ov.round(3))

print("\n========== 부호 일관성 (6개 모델 중 d > 0 개수) ==========")
for nm in VARIANTS:
    col = f'd_{nm}'
    n_pos = int((ov[col] > 0).sum())
    lin = int((ov.loc[['LinearRegression', 'ElasticNet', 'SVR_Linear', 'SVR_RBF'], col] > 0).sum())
    print(f"  {nm:14s} {n_pos}/6   (선형·커널 4개 중 {lin})")
print("  -> 네 변형은 서로 r>0.95로 닮았다. 한 변형만 + 면 신호가 아니라 노이즈로 읽는다.")

print("\n========== 성능 순위 (Overall RMSE) ==========")
display(res[res.Scope == 'Overall'].sort_values('RMSE')
        [['Data Type', 'Model', 'RMSE', 'R2', 'r', 'p']]
        .round(4).reset_index(drop=True).head(12))

In [ ]:
# ==========================================================
# 5. Active vs Sham — 치우침과 흩어짐을 나눠서 본다
#
#  group을 모델에 넣지 않았으면(USE_TREATMENT_GROUP=False) 예측값은 군 라벨과 무관하다.
#  따라서 라벨을 섞어 귀무분포를 만들 수 있다(재학습 불필요).
#
#  bias  = 평균잔차. group을 빼면 여기에 치료효과가 그대로 남는다.
#  RMSE  = sqrt(흩어짐^2 + bias^2) 이므로 bias를 같이 봐야 해석된다.
#  R2    = 군마다 분산이 달라 군 간 비교 불가 (4번 셀에만 두고 여기서는 쓰지 않는다).
# ==========================================================
N_PERM = 5000
act = group == 'Active'
n_act = int(act.sum())

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

rows = []
for (dt, name), preds in predictions.items():
    yp = np.array(preds)
    r_a, r_s = yp[act] - y_true[act], yp[~act] - y_true[~act]
    obs = rmse(yp[act], y_true[act]) - rmse(yp[~act], y_true[~act])

    if not USE_TREATMENT_GROUP:      # 순열검정은 group-blind 모델에서만 유효
        resid_all = yp - y_true
        null = np.empty(N_PERM)
        for b in range(N_PERM):
            idx = rng.permutation(len(y_true))
            ra, rs = resid_all[idx[:n_act]], resid_all[idx[n_act:]]
            null[b] = np.sqrt(np.mean(ra ** 2)) - np.sqrt(np.mean(rs ** 2))
        p_perm = float(np.mean(np.abs(null) >= abs(obs)))
    else:
        p_perm = np.nan

    rows.append({'Data Type': dt, 'Model': name,
                 'RMSE_Act': rmse(yp[act], y_true[act]),
                 'RMSE_Sham': rmse(yp[~act], y_true[~act]),
                 'dRMSE': obs, 'p_perm': p_perm,
                 'bias_Act': r_a.mean(), 'bias_Sham': r_s.mean(),
                 'bias_diff': r_a.mean() - r_s.mean(),
                 'RMSE_Act_nobias': float(np.sqrt(np.mean((r_a - r_a.mean()) ** 2))),
                 'RMSE_Sham_nobias': float(np.sqrt(np.mean((r_s - r_s.mean()) ** 2))),
                 'r_Act': pearsonr(y_true[act], yp[act])[0],
                 'r_Sham': pearsonr(y_true[~act], yp[~act])[0]})

grp = pd.DataFrame(rows)
grp['Data Type'] = pd.Categorical(grp['Data Type'], DATA_TYPES)
grp['Model'] = pd.Categorical(grp['Model'], list(models))
grp = grp.sort_values(['Data Type', 'Model']).reset_index(drop=True)

print("bias_diff = Active 평균잔차 - Sham 평균잔차")
print("  group을 뺀 모델에서는 이 값이 ANCOVA의 치료효과 추정치와 거의 같아진다(약 -1.0점).")
print("  즉 치우침은 잡음이 아니라 '치료가 들었다'는 신호다.\n")
print("dRMSE = Active RMSE - Sham RMSE (음수면 Active에서 오차가 작음)")
print("  치우침이 섞여 있으므로 *_nobias 열과 r_* 열을 같이 볼 것.")
print("p_perm = 군 라벨 5000회 무작위 재배정으로 만든 귀무분포 기준 p값\n")
display(grp.round(3))

In [ ]:
# ==========================================================
# 6. VISUALIZATION — 전체 RMSE
# ==========================================================
sns.set_theme(style="ticks")
plt.rcParams['font.sans-serif'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ov_long = res[res.Scope == 'Overall']
fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(data=ov_long, x='Model', y='RMSE', hue='Data Type', hue_order=DATA_TYPES, ax=ax)
ax.axhline(baseline_rmse, color='black', ls='--', lw=1)
ax.text(0.01, baseline_rmse, ' 평균만 예측', va='bottom', fontsize=9)
clin = ov_long.loc[ov_long['Data Type'] == 'Clinical_ONLY', 'RMSE'].min()
ax.axhline(clin, color='#457B9D', ls=':', lw=1.2)
ax.text(0.01, clin, ' Clinical 최저', va='bottom', fontsize=9, color='#457B9D')
ax.set_title(f'Overall LOOCV RMSE (n={len(y_true)}) — 낮을수록 좋음')
ax.tick_params(axis='x', rotation=25)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# 7. 산점도 — Active vs Sham (보고 싶은 조합만 지정)
# ==========================================================
PLOT = [('Coup_glob_S', 'ElasticNet'), ('Coup_reg_S', 'SVR_RBF')]   # 필요에 맞게 수정
palette = {'Active rTMS': '#D95F02', 'Sham': '#7570B3'}
grp_full = np.where(group == 'Active', 'Active rTMS', 'Sham')

for dt, name in PLOT:
    yp = np.array(predictions[(dt, name)])
    pdf = pd.DataFrame({'Actual': y_true, 'Predicted': yp, 'Group': grp_full})
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.3), sharex=True, sharey=True, dpi=110)
    lo = min(pdf.Actual.min(), pdf.Predicted.min()) - 2
    hi = max(pdf.Actual.max(), pdf.Predicted.max()) + 2
    for i, g in enumerate(['Active rTMS', 'Sham']):
        ax = axes[i]
        sub = pdf[pdf.Group == g]
        r, pv = pearsonr(sub.Actual, sub.Predicted)
        bias = (sub.Predicted - sub.Actual).mean()
        rm = np.sqrt(np.mean((sub.Actual - sub.Predicted) ** 2))
        rm_nb = np.sqrt(np.mean((sub.Predicted - sub.Actual - bias) ** 2))
        ax.plot([lo, hi], [lo, hi], color='#A0A0A0', ls='--', lw=1.3)
        sns.scatterplot(data=sub, x='Actual', y='Predicted', color=palette[g], s=65,
                        edgecolor='black', linewidth=0.5, ax=ax)
        sns.regplot(data=sub, x='Actual', y='Predicted', ax=ax, scatter=False,
                    color=palette[g], line_kws={'linewidth': 2})
        ax.set_title(f'{g}  (n={len(sub)})', fontweight='bold')
        ax.text(0.05, 0.72,
                f"r = {r:.3f} (p = {pv:.3f})\n치우침 = {bias:+.3f}\n"
                f"RMSE = {rm:.2f}\n치우침 제거 = {rm_nb:.2f}",
                transform=ax.transAxes, fontsize=9.5,
                bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#CCC', alpha=0.9))
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_xlabel('Actual VAS $T_1$')
        if i == 0:
            ax.set_ylabel('Predicted VAS $T_1$')
    fig.suptitle(f'{name}  /  {dt}', fontweight='bold', y=1.02)
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================================
# 8. 하이퍼파라미터 안정성
# ==========================================================
for (dt, name), plist in best_params_log.items():
    vc = pd.Series(plist).value_counts()
    share = vc.iloc[0] / vc.sum()
    flag = '' if share >= 0.5 else '   <-- 불안정'
    print(f"[{dt} / {name}] 고유 {len(vc)}개, 최빈 {share:.0%}{flag}")

In [ ]:
# ==========================================================
# 9. coupling 4변형 기술통계 (vas를 쓰지 않음 — 예측과 무관)
# ==========================================================
comp = pd.DataFrame({'glob_P': X_gP[:, 0], 'glob_S': X_gS[:, 0],
                     'reg_P_mean': X_rP.mean(1), 'reg_S_mean': X_rS.mean(1)})
print('=== 변형끼리의 피험자 간 상관 ===')
display(comp.corr().round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
axes[0].scatter(comp.glob_P, comp.glob_S, color='#6B4FBB', s=45, edgecolor='white')
axes[0].set_xlabel('전역 Pearson')
axes[0].set_ylabel('전역 Spearman')
axes[0].set_title(f'전역 두 방식  r = {comp.glob_P.corr(comp.glob_S):.3f}')

axes[1].hist([X_rP.ravel(), X_rS.ravel()], bins=40,
             label=['지역 Pearson', '지역 Spearman'], color=['#2A9D8F', '#E9C46A'])
axes[1].axvline(0, color='black', lw=1)
axes[1].legend()
axes[1].set_title('지역 coupling 분포 (44명 x 68 ROI)')

roi_mean = pd.Series(X_rS.mean(0), index=ROIS).sort_values()
axes[2].barh(roi_mean.index[-12:], roi_mean.values[-12:], color='#457B9D')
axes[2].barh(roi_mean.index[:12], roi_mean.values[:12], color='#E63946')
axes[2].tick_params(axis='y', labelsize=7)
axes[2].set_title('지역 Spearman 상·하위 12 ROI')
plt.tight_layout()
plt.show()

print('선행연구(SC-FC)에서는 감각·시각 피질이 높고 연합 피질(DMN)이 낮다고 보고됐다.')
print('다만 그 문헌의 구조는 확산 MRI 연결이라 MS-FC에 그대로 적용된다는 보장은 없다. 참고로만 볼 것.')

## 결과 읽는 법

1. **일관성이 첫 번째 기준.** 네 변형은 서로 r > 0.95다. 따라서 **셋 이상에서 같은 방향**이어야
   신호로 본다. 한 변형에서만 이득이 보이면 노이즈일 가능성이 높다.
2. **전역 vs 지역**이 진짜 비교다. 전역은 피처 1개, 지역은 PCA 3개다.
   전역이 지역만큼 하면 **더 단순한 쪽을 택한다** (n=44에서는 단순함이 곧 안정성).
3. **Pearson vs Spearman**은 민감도 확인이다. 둘이 갈리면 극단값(|r| > 0.9인 MSN edge 7.9%)의
   영향을 의심하고, 그때는 Spearman 쪽을 믿는다.
4. **Active/Sham은 5번 셀에서** `bias`와 `*_nobias`를 함께 본다.
   - `bias_diff`가 약 −1.0 → 치료효과가 잔차에 남은 것(정상, 기대된 결과)
   - `dRMSE`가 음수라도 그중 일부는 치우침 탓 → `RMSE_*_nobias` 차이와 `r_*`로 확인
   - `p_perm`은 군 라벨을 섞어 만든 귀무분포 기준 p값 (group-blind 모델에서만 유효)
5. **하지 말 것**: 네 변형 중 좋은 것만 보고하기. PCA 성분 수를 바꿔가며 돌리기.
   9번 셀에서 눈에 띄는 ROI를 골라 피처로 쓰기(같은 44명으로 고르면 leakage).